In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from patsy import dmatrix
from sklearn.metrics import mean_squared_error
from statsmodels.tsa.arima.model import ARIMA
from scipy.special import logit, expit
import statsmodels.api as sm
import matplotlib.pyplot as plt
from prophet import Prophet
from scipy.stats import norm 

c:\Users\psy09\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Outcome: AMD
Data: GBD 2021
Step: 2. prediction_model
weze_code_ver

```
step1. PAF 계산
step2. Merge and filtering
step3. Omega value에 따른 ARC and prediction
step4. Omega value에 따른 MR-BRT and PAF
step5. RMSE
step6. Ensemble model
step7. Normalized and scaling

In [29]:
# load
sev_df=pd.read_csv("C:/Users/psy09/Desktop/Lab/5.AMD(GBD_database)/2.revision/prediction_model/data/covariates/AMD_SEV(forecast).csv") 
rr_df=pd.read_csv("C:/Users/psy09/Desktop/Lab/5.AMD(GBD_database)/2.revision/prediction_model/data/covariates/AMD_RR(final).csv") 
prevalence_df=pd.read_csv("C:/Users/psy09/Desktop/Lab/5.AMD(GBD_database)/2.revision/prediction_model/data/amd2021/amd_total.csv") 
pop_df=pd.read_csv("C:/Users/psy09/Desktop/Lab/5.AMD(GBD_database)/2.revision/prediction_model/data/population/pop_2050.csv")
sdi_df=pd.read_csv('C:/Users/psy09/Desktop/Lab/5.AMD(GBD_database)/2.revision/prediction_model/data/covariates/SDI(forecast).csv')

In [30]:
# merge sev and rr
sev_df = sev_df.rename(columns={'val': 'sev_val', 'lower': 'sev_lower', 'upper': 'sev_upper'})
merged_df = pd.merge(
    sev_df,
    rr_df[['location_name', 'sex_id', 'age_name', 'year', 'smoke_cate', 'RR', 'RR_lower', 'RR_upper']],
    on=['location_name', 'sex_id', 'age_name', 'year', 'smoke_cate'], 
    how='left'
)
merged_df

,Unnamed: 0,year,sev_val,sev_lower,sev_upper,location_name,sex_id,age_name,age_id,sex_name,smoke_cate,RR,RR_lower,RR_upper
0,0,1990,40.182944,40.144013,40.221123,Global,1,45-49 years,14,Male,0 Cigarettes Per Day,1.000000,1.000000,1.000000
1,1,1991,39.908633,39.871321,39.943873,Global,1,45-49 years,14,Male,0 Cigarettes Per Day,1.000000,1.000000,1.000000
2,2,1992,39.738234,39.698750,39.774672,Global,1,45-49 years,14,Male,0 Cigarettes Per Day,1.000000,1.000000,1.000000
3,3,1993,39.620800,39.583497,39.658905,Global,1,45-49 years,14,Male,0 Cigarettes Per Day,1.000000,1.000000,1.000000
4,4,1994,39.617769,39.581171,39.656442,Global,1,45-49 years,14,Male,0 Cigarettes Per Day,1.000000,1.000000,1.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
261883,32731,2046,1.118779,-2.124605,4.366699,Tropical Latin America,2,95+ years,235,Female,Mean Per Day,2.207143,1.418571,3.208571
261884,32732,2047,1.022674,-2.381158,4.478648,Tropical Latin America,2,95+ years,235,Female,Mean Per Day,2.207143,1.418571,3.208571
261885,32733,2048,0.968613,-2.601183,4.564697,Tropical Latin America,2,95+ years,235,Female,Mean Per Day,2.207143,1.418571,3.208571
261886,32734,2049,0.873432,-2.875856,4.657567,Tropical Latin America,2,95+ years,235,Female,Mean Per Day,2.207143,1.418571,3.208571


In [31]:
# PAF and scaler + clipping
def paf_and_scaler_clipped(df):
    df['sev_val'] = df['sev_val'].clip(lower=0, upper=1)
    # PAF 계산: 1 - (1 / (SEV * (RR - 1) + 1))
    df['PAF'] = 1 - (1 / (df['sev_val'] * (df['RR'] - 1) + 1))
    # Scaler 계산: 1 / (1 - PAF)
    df['scaler'] = 1 / (1 - df['PAF'])
    
    return df

In [32]:
paf_results = paf_and_scaler_clipped(merged_df)
paf_results = paf_results.loc[:, ~paf_results.columns.str.contains('^Unnamed')]
paf_results.to_csv('C:/Users/psy09/Desktop/Lab/5.AMD(GBD_database)/2.revision/prediction_model/data/paf_results.csv')
paf_results

,year,sev_val,sev_lower,sev_upper,location_name,sex_id,age_name,age_id,sex_name,smoke_cate,RR,RR_lower,RR_upper,PAF,scaler
0,1990,1.000000,40.144013,40.221123,Global,1,45-49 years,14,Male,0 Cigarettes Per Day,1.000000,1.000000,1.000000,0.000000,1.000000
1,1991,1.000000,39.871321,39.943873,Global,1,45-49 years,14,Male,0 Cigarettes Per Day,1.000000,1.000000,1.000000,0.000000,1.000000
2,1992,1.000000,39.698750,39.774672,Global,1,45-49 years,14,Male,0 Cigarettes Per Day,1.000000,1.000000,1.000000,0.000000,1.000000
3,1993,1.000000,39.583497,39.658905,Global,1,45-49 years,14,Male,0 Cigarettes Per Day,1.000000,1.000000,1.000000,0.000000,1.000000
4,1994,1.000000,39.581171,39.656442,Global,1,45-49 years,14,Male,0 Cigarettes Per Day,1.000000,1.000000,1.000000,0.000000,1.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
261883,2046,1.000000,-2.124605,4.366699,Tropical Latin America,2,95+ years,235,Female,Mean Per Day,2.207143,1.418571,3.208571,0.546926,2.207143
261884,2047,1.000000,-2.381158,4.478648,Tropical Latin America,2,95+ years,235,Female,Mean Per Day,2.207143,1.418571,3.208571,0.546926,2.207143
261885,2048,0.968613,-2.601183,4.564697,Tropical Latin America,2,95+ years,235,Female,Mean Per Day,2.207143,1.418571,3.208571,0.539012,2.169254
261886,2049,0.873432,-2.875856,4.657567,Tropical Latin America,2,95+ years,235,Female,Mean Per Day,2.207143,1.418571,3.208571,0.513230,2.054357


In [33]:
# PAF min max
min_paf = paf_results['PAF'].min()
max_paf = paf_results['PAF'].max()
print(f"PAF 최소값: {min_paf}")
print(f"PAF 최대값: {max_paf}")
# SEV and RR min max
print("SEV 최소값:", paf_results['sev_val'].min())
print("SEV 최대값:", paf_results['sev_val'].max())
print("RR 최소값:", paf_results['RR'].min())
print("RR 최대값:", paf_results['RR'].max())
avg_paf_by_group = paf_results.groupby(['age_name', 'sex_name'])['PAF'].mean()
print(avg_paf_by_group)

PAF 최소값: 0.0
PAF 최대값: 0.6753246753246753
SEV 최소값: 0.0
SEV 최대값: 1.0
RR 최소값: 1.0
RR 최대값: 3.08
age_name     sex_name
45-49 years  Female      0.482319
             Male        0.484928
50-54 years  Female      0.482908
             Male        0.484928
55-59 years  Female      0.484928
             Male        0.484928
60-64 years  Female      0.484489
             Male        0.484928
65-69 years  Female      0.479729
             Male        0.484928
70-74 years  Female      0.472918
             Male        0.484928
75-79 years  Female      0.463513
             Male        0.484928
80-84 years  Female      0.456034
             Male        0.484928
85-89 years  Female      0.457549
             Male        0.484928
90-94 years  Female      0.460724
             Male        0.484928
95+ years    Female      0.451972
             Male        0.483342
All ages     Female      0.483444
             Male        0.484928
Name: PAF, dtype: float64


In [34]:
# PAF filtering
filtered_df = paf_results[
    (paf_results['location_name'] == 'Global') &
    (paf_results['age_name'] == 'All ages') &
    (paf_results['smoke_cate'] == 'Mean Per Day')
]
filtered_df = filtered_df.drop(columns=['Unnamed: 0', 'val', 'lower', 'upper'], errors='ignore')
print("Filtered PAF Data:")
print(filtered_df.tail())
# SDI filtering
filtered_sdi = sdi_df[
    (sdi_df['location_name'] == 'Global') &
    (sdi_df['age_name'] == 'All ages')
]
filtered_sdi = filtered_sdi.drop(columns=['Unnamed: 0', 'Unnamed: 0.1', 'Unnamed: 0.2'], errors='ignore')
print("Filtered SDI Data:")
print(filtered_sdi.tail())
# AMD Prevalence filtering
filtered_amd = prevalence_df[
    (prevalence_df['location_name'] == 'Global') &
    (prevalence_df['age_name'] == 'All ages')
]
filtered_amd = filtered_amd.drop(columns=['Unnamed: 0', 'age_id', 'location_id'], errors='ignore')
print("Filtered AMD Data:")
print(filtered_amd.tail())

# 2022-2050 data
years_to_add = list(range(2022, 2051))
expanded_data = []

for _, row in filtered_amd[['location_name', 'sex_name', 'age_name', 'sex_id']].drop_duplicates().iterrows():
    for year in years_to_add:
        expanded_data.append({
            'location_name': row['location_name'],
            'sex_id': row['sex_id'],
            'sex_name': row['sex_name'],
            'age_name': row['age_name'],
            'year': year,
            'val': None,  # NaN으로 설정
            'upper': None,  # NaN으로 설정
            'lower': None  # NaN으로 설정
        })

expanded_df = pd.DataFrame(expanded_data)

expanded_df = expanded_df[~expanded_df[['location_name', 'sex_id', 'sex_name', 'age_name', 'year']].isin(
    filtered_amd[['location_name', 'sex_id', 'sex_name', 'age_name', 'year']]
).all(axis=1)]
full_amd = pd.concat([filtered_amd, expanded_df], ignore_index=True)
full_amd = full_amd.drop_duplicates()

# merge
merged_df = full_amd.merge(
    filtered_df[['location_name', 'sex_id', 'age_name', 'year', 'PAF', 'scaler', 'smoke_cate']],
    on=['location_name', 'sex_id', 'age_name', 'year'],
    how='left'
)
merged_df = merged_df.merge(
    filtered_sdi[['location_name', 'year', 'sdi_val']],
    on=['location_name', 'year'],
    how='left'
)

Filtered PAF Data:
        year  sev_val  sev_lower  sev_upper location_name  sex_id  age_name  \
230387  2046      1.0   1.899346   3.135456        Global       2  All ages   
230388  2047      1.0   1.767605   3.064938        Global       2  All ages   
230389  2048      1.0   1.560215   2.940272        Global       2  All ages   
230390  2049      1.0   1.410673   2.849086        Global       2  All ages   
230391  2050      1.0   1.257906   2.789511        Global       2  All ages   

        age_id sex_name    smoke_cate        RR  RR_lower  RR_upper       PAF  \
230387      22   Female  Mean Per Day  2.207143  1.418571  3.208571  0.546926   
230388      22   Female  Mean Per Day  2.207143  1.418571  3.208571  0.546926   
230389      22   Female  Mean Per Day  2.207143  1.418571  3.208571  0.546926   
230390      22   Female  Mean Per Day  2.207143  1.418571  3.208571  0.546926   
230391      22   Female  Mean Per Day  2.207143  1.418571  3.208571  0.546926   

          scaler  


C:\Users\psy09\AppData\Local\Temp\ipykernel_22620\4294497304.py:49: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  full_amd = pd.concat([filtered_amd, expanded_df], ignore_index=True)


In [35]:
merged_df = merged_df.drop_duplicates(subset=['location_name', 'sex_id', 'age_name', 'year', 'val', 'upper', 'lower', 'PAF', 'scaler', 'smoke_cate', 'sdi_val'])
print(merged_df.head())

  location_name  sex_id sex_name  age_name  year       val     upper  \
0        Global       1     Male  All ages  1990  4.039887  4.226359   
2        Global       2   Female  All ages  1990  4.404975  4.572757   
4        Global       1     Male  All ages  1991  4.054218  4.240694   
6        Global       2   Female  All ages  1991  4.422290  4.590783   
8        Global       1     Male  All ages  1992  4.067776  4.254614   

      lower       PAF    scaler    smoke_cate   sdi_val  
0  3.858314  0.546926  2.207143  Mean Per Day  0.525767  
2  4.226861  0.546926  2.207143  Mean Per Day  0.525767  
4  3.870502  0.546926  2.207143  Mean Per Day  0.530014  
6  4.244331  0.546926  2.207143  Mean Per Day  0.530014  
8  3.881820  0.546926  2.207143  Mean Per Day  0.534287  


In [36]:
# omega 값 목록
omega_values = [0.0, 0.5, 1.0, 1.5, 2.0, 2.5]

# 데이터프레임을 6번 복제하여 omega 값을 추가
merged_df_omega = merged_df.loc[merged_df.index.repeat(len(omega_values))].reset_index(drop=True)

# omega 값을 각 행에 순차적으로 할당
merged_df_omega['omega'] = np.tile(omega_values, len(merged_df_omega) // len(omega_values) + 1)[:len(merged_df_omega)]
merged_df_omega

,location_name,sex_id,sex_name,age_name,year,val,upper,lower,PAF,scaler,smoke_cate,sdi_val,omega
0,Global,1,Male,All ages,1990,4.039887,4.226359,3.858314,0.546926,2.207143,Mean Per Day,0.525767,0.0
1,Global,1,Male,All ages,1990,4.039887,4.226359,3.858314,0.546926,2.207143,Mean Per Day,0.525767,0.5
2,Global,1,Male,All ages,1990,4.039887,4.226359,3.858314,0.546926,2.207143,Mean Per Day,0.525767,1.0
3,Global,1,Male,All ages,1990,4.039887,4.226359,3.858314,0.546926,2.207143,Mean Per Day,0.525767,1.5
4,Global,1,Male,All ages,1990,4.039887,4.226359,3.858314,0.546926,2.207143,Mean Per Day,0.525767,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
739,Global,2,Female,All ages,2050,NaN,NaN,NaN,0.546926,2.207143,Mean Per Day,0.809357,0.5
740,Global,2,Female,All ages,2050,NaN,NaN,NaN,0.546926,2.207143,Mean Per Day,0.809357,1.0
741,Global,2,Female,All ages,2050,NaN,NaN,NaN,0.546926,2.207143,Mean Per Day,0.809357,1.5
742,Global,2,Female,All ages,2050,NaN,NaN,NaN,0.546926,2.207143,Mean Per Day,0.809357,2.0


In [37]:
merged_df_omega.to_csv('C:/Users/psy09/Desktop/Lab/5.AMD(GBD_database)/2.revision/prediction_model/data/merged_df.csv')

Omega version

In [38]:
# 1. Calcaulate ARC (Annualized Rate of Change) with omega
def calculate_actual_arc(data):
    grouped = data.groupby(['location_name', 'sex_name', 'age_name', 'smoke_cate'])

    def linear_regression(group):
        model = LinearRegression()
        model.fit(
            group['year'].values.reshape(-1, 1),  # 독립 변수: year
            group['val'].fillna(0).values         # 종속 변수: prevalence_rate
        )
        return model.coef_[0]

    arc_results_actual = grouped.apply(linear_regression).reset_index()
    arc_results_actual.columns = ['location_name', 'sex_name', 'age_name', 'smoke_cate', 'actual_arc']
    arc_results_actual['actual_arc'] = arc_results_actual['actual_arc'] * 100  # ARC를 퍼센트로
    return arc_results_actual

In [39]:
# 2. ARC-based prediction with omega
def calculate_arc_with_omega(data):
    grouped = data.groupby(['location_name', 'sex_name', 'age_name', 'smoke_cate', 'omega'])

    def weighted_regression(group):
        omega = group['omega'].iloc[0]
        weights = np.exp(-omega * (2021 - group['year']))
        model = LinearRegression()
        model.fit(
            group['year'].values.reshape(-1, 1),
            group['val'].fillna(0).values,
            sample_weight=weights
        )
        return model.coef_[0]

    arc_results = grouped.apply(weighted_regression).reset_index()
    arc_results.columns = ['location_name', 'sex_name', 'age_name', 'smoke_cate', 'omega', 'arc']
    arc_results['arc'] = arc_results['arc'] * 100
    return arc_results

In [40]:
def add_actual_and_forecast_arc(data, start_year=2022, end_year=2050):
    # 1990-2021의 ARC
    actual_arc_results = calculate_actual_arc(data[data['year'] <= 2021])
    # 2022-2050 ARC
    arc_results_forecast = calculate_arc_with_omega(data[data['year'] <= 2021])

    data_with_actual_arc = data.merge(
        actual_arc_results, 
        on=['location_name', 'sex_name', 'age_name', 'smoke_cate'], 
        how='left'
    )
    data_with_full_arc = data_with_actual_arc.merge(
        arc_results_forecast, 
        on=['location_name', 'sex_name', 'age_name', 'smoke_cate', 'omega'], 
        how='left'
    )

    data_with_full_arc = data_with_full_arc.loc[:, ~data_with_full_arc.columns.duplicated()]
    forecast_data = []
    merged_data = data_with_full_arc

    for year in range(start_year, end_year + 1):
        forecast_year = merged_data[merged_data['year'] == 2021].copy()
        forecast_year['year'] = year
        forecast_year['val'] += (year - 2021) * forecast_year['arc'] / 100
        forecast_data.append(forecast_year)

    forecast_df = pd.concat(forecast_data)
    final_data = pd.concat([data_with_full_arc[data_with_full_arc['year'] <= 2021], forecast_df], ignore_index=True)
    return final_data

In [41]:
forecast_arc = add_actual_and_forecast_arc(merged_df_omega)
forecast_arc

C:\Users\psy09\AppData\Local\Temp\ipykernel_22620\3441915159.py:13: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  arc_results_actual = grouped.apply(linear_regression).reset_index()
C:\Users\psy09\AppData\Local\Temp\ipykernel_22620\890403230.py:16: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  arc_results = grouped.apply(weighted_regression).reset_index()


,location_name,sex_id,sex_name,age_name,year,val,upper,lower,PAF,scaler,smoke_cate,sdi_val,omega,actual_arc,arc
0,Global,1,Male,All ages,1990,4.039887,4.226359,3.858314,0.546926,2.207143,Mean Per Day,0.525767,0.0,1.348253,1.348253
1,Global,1,Male,All ages,1990,4.039887,4.226359,3.858314,0.546926,2.207143,Mean Per Day,0.525767,0.5,1.348253,1.884812
2,Global,1,Male,All ages,1990,4.039887,4.226359,3.858314,0.546926,2.207143,Mean Per Day,0.525767,1.0,1.348253,1.274145
3,Global,1,Male,All ages,1990,4.039887,4.226359,3.858314,0.546926,2.207143,Mean Per Day,0.525767,1.5,1.348253,0.472861
4,Global,1,Male,All ages,1990,4.039887,4.226359,3.858314,0.546926,2.207143,Mean Per Day,0.525767,2.0,1.348253,-0.201062
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1087,Global,2,Female,All ages,2050,5.363738,4.974764,4.602438,0.546926,2.207143,Mean Per Day,0.670289,0.5,1.133701,2.002587
1088,Global,2,Female,All ages,2050,5.285198,4.974764,4.602438,0.546926,2.207143,Mean Per Day,0.670289,1.0,1.133701,1.731762
1089,Global,2,Female,All ages,2050,5.155034,4.974764,4.602438,0.546926,2.207143,Mean Per Day,0.670289,1.5,1.133701,1.282920
1090,Global,2,Female,All ages,2050,5.044800,4.974764,4.602438,0.546926,2.207143,Mean Per Day,0.670289,2.0,1.133701,0.902801


In [42]:
merge_keys = ['location_name', 'sex_name', 'age_name', 'year', 'smoke_cate', 'omega']

# merged_df_omega에 forecast_arc 데이터를 병합
merged_df_omega = merged_df_omega.merge(
    forecast_arc[merge_keys + ['arc', 'val', 'upper', 'lower']],
    on=merge_keys,
    how='left',
    suffixes=('', '_forecast')
)
columns_to_update = ['val', 'upper', 'lower']
for col in columns_to_update:
    merged_df_omega[col] = merged_df_omega[f'{col}_forecast'].combine_first(merged_df_omega[col])

merged_df_omega.drop(columns=[f'{col}_forecast' for col in columns_to_update], inplace=True)
forecast_arc_updated = merged_df_omega.copy()
forecast_arc_updated = forecast_arc.iloc[:, 1:]
forecast_arc_updated.to_csv('C:/Users/psy09/Desktop/Lab/5.AMD(GBD_database)/2.revision/prediction_model/data/forecast_arc.csv', index=False)
forecast_arc_updated

,sex_id,sex_name,age_name,year,val,upper,lower,PAF,scaler,smoke_cate,sdi_val,omega,actual_arc,arc
0,1,Male,All ages,1990,4.039887,4.226359,3.858314,0.546926,2.207143,Mean Per Day,0.525767,0.0,1.348253,1.348253
1,1,Male,All ages,1990,4.039887,4.226359,3.858314,0.546926,2.207143,Mean Per Day,0.525767,0.5,1.348253,1.884812
2,1,Male,All ages,1990,4.039887,4.226359,3.858314,0.546926,2.207143,Mean Per Day,0.525767,1.0,1.348253,1.274145
3,1,Male,All ages,1990,4.039887,4.226359,3.858314,0.546926,2.207143,Mean Per Day,0.525767,1.5,1.348253,0.472861
4,1,Male,All ages,1990,4.039887,4.226359,3.858314,0.546926,2.207143,Mean Per Day,0.525767,2.0,1.348253,-0.201062
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1087,2,Female,All ages,2050,5.363738,4.974764,4.602438,0.546926,2.207143,Mean Per Day,0.670289,0.5,1.133701,2.002587
1088,2,Female,All ages,2050,5.285198,4.974764,4.602438,0.546926,2.207143,Mean Per Day,0.670289,1.0,1.133701,1.731762
1089,2,Female,All ages,2050,5.155034,4.974764,4.602438,0.546926,2.207143,Mean Per Day,0.670289,1.5,1.133701,1.282920
1090,2,Female,All ages,2050,5.044800,4.974764,4.602438,0.546926,2.207143,Mean Per Day,0.670289,2.0,1.133701,0.902801


In [43]:
# 3. MR-BRT model setting
def run_mr_brt(data, sdi_col, log_rate_col, year_col):
    # SDI 값의 최소/최대 범위에 따른 knots 생성
    sdi_min, sdi_max = data[sdi_col].min(), data[sdi_col].max()
    knots = np.linspace(sdi_min + 0.01 * (sdi_max - sdi_min), 
                        sdi_max - 0.01 * (sdi_max - sdi_min), 
                        3)
    # 스플라인 생성 (dmatrix 수정)
    spline = dmatrix(
        f"bs({sdi_col}, knots={list(knots)}, degree=3, include_intercept=True)",
        data,
        return_type='dataframe'
    )
    # 스플라인 기반 예측 (MR-BRT 첫 번째 단계)
    model1 = LinearRegression()
    model1.fit(spline, data[log_rate_col])
    data['spline_pred'] = model1.predict(spline)
    # 잔차 계산 및 시간 추세 학습 (MR-BRT 두 번째 단계)
    data['residuals'] = data[log_rate_col] - data['spline_pred']
    model2 = LinearRegression()
    model2.fit(data[[year_col]], data['residuals'])
    data['time_trend'] = model2.predict(data[[year_col]])

    data['mr_brt_val'] = data['spline_pred'] + data['time_trend']
    return data

In [44]:
# run MR-BRT model with omega
def run_mr_brt_with_omega(data, sdi_col, log_rate_col, year_col, omega_col):
    if omega_col not in data.columns:
        raise ValueError(f"Column {omega_col} not found in the data!")

    results = []
    for omega in data[omega_col].unique():
        omega_data = data[data[omega_col] == omega].copy()
        print(f"Running MR-BRT for omega = {omega}...")
        try:
            omega_result = run_mr_brt(omega_data, sdi_col, log_rate_col, year_col)
            omega_result['omega'] = omega 
            results.append(omega_result)
        except Exception as e:
            print(f"Error running MR-BRT for omega {omega}: {e}")
            continue

    final_results = pd.concat(results, axis=0).reset_index(drop=True)
    return final_results

In [45]:
mr_brt_results = run_mr_brt_with_omega(
    data=forecast_arc_updated,          # 사용 데이터
    sdi_col='sdi_val',            # SDI 열 이름
    log_rate_col='val',      # 로그 변환된 비율 열 이름
    year_col='year',              # 연도 열 이름
    omega_col='omega'             # Omega 열 이름
)

Running MR-BRT for omega = 0.0...
Running MR-BRT for omega = 0.5...
Running MR-BRT for omega = 1.0...
Running MR-BRT for omega = 1.5...
Running MR-BRT for omega = 2.0...
Running MR-BRT for omega = 2.5...


In [46]:
mr_brt_results.to_csv('C:/Users/psy09/Desktop/Lab/5.AMD(GBD_database)/2.revision/prediction_model/data/forcast_mrbrt.csv')
mr_brt_results

,sex_id,sex_name,age_name,year,val,upper,lower,PAF,scaler,smoke_cate,sdi_val,omega,actual_arc,arc,spline_pred,residuals,time_trend,mr_brt_val
0,1,Male,All ages,1990,4.039887,4.226359,3.858314,0.546926,2.207143,Mean Per Day,0.525767,0.0,1.348253,1.348253,4.222519,-0.182632,-0.077465,4.145054
1,2,Female,All ages,1990,4.404975,4.572757,4.226861,0.546926,2.207143,Mean Per Day,0.525767,0.0,1.133701,1.133701,4.222519,0.182456,-0.077465,4.145054
2,1,Male,All ages,1991,4.054218,4.240694,3.870502,0.546926,2.207143,Mean Per Day,0.530014,0.0,1.348253,1.348253,4.260208,-0.205990,-0.075257,4.184951
3,2,Female,All ages,1991,4.422290,4.590783,4.244331,0.546926,2.207143,Mean Per Day,0.530014,0.0,1.133701,1.133701,4.260208,0.162082,-0.075257,4.184951
4,1,Male,All ages,1992,4.067776,4.254614,3.881820,0.546926,2.207143,Mean Per Day,0.534287,0.0,1.348253,1.348253,4.256866,-0.189091,-0.073050,4.183816
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1087,2,Female,All ages,2049,4.958793,4.974764,4.602438,0.546926,2.207143,Mean Per Day,0.670289,2.5,1.133701,0.627877,4.618927,0.339866,-0.001658,4.617269
1088,1,Male,All ages,2050,4.263813,4.665762,4.275678,0.546926,2.207143,Mean Per Day,0.665228,2.5,1.348253,-0.691483,4.620178,-0.356365,-0.001730,4.618448
1089,1,Male,All ages,2050,4.263813,4.665762,4.275678,0.546926,2.207143,Mean Per Day,0.670289,2.5,1.348253,-0.691483,4.618927,-0.355114,-0.001730,4.617197
1090,2,Female,All ages,2050,4.965072,4.974764,4.602438,0.546926,2.207143,Mean Per Day,0.665228,2.5,1.133701,0.627877,4.620178,0.344893,-0.001730,4.618448


In [47]:
def adjust_mr_brt_values(data, year_col, val_col, mr_brt_col):
    # 1990-2021: mr_brt_val을 val 값으로 대체
    actual_period = (data[year_col] <= 2021)
    future_period = (data[year_col] > 2021)

    # 2021년의 val과 mr_brt_val 차이 계산
    adjustment = data.loc[data[year_col] == 2021, val_col].values[0] - data.loc[data[year_col] == 2021, mr_brt_col].values[0]
    
    # 1990-2021: mr_brt_val을 val로 대체
    data.loc[actual_period, mr_brt_col] = data.loc[actual_period, val_col]

    # 2022-2050: 2021년 차이를 모든 mr_brt_val 값에 더함
    data.loc[future_period, mr_brt_col] += adjustment

    return data

# 조정 실행
mr_brt_results_adjusted = adjust_mr_brt_values(
    data=mr_brt_results,
    year_col='year',
    val_col='val',
    mr_brt_col='mr_brt_val'
)

mr_brt_results_adjusted.to_csv('C:/Users/psy09/Desktop/Lab/5.AMD(GBD_database)/2.revision/prediction_model/data/forcast_mrbrt.csv')
mr_brt_results_adjusted

,sex_id,sex_name,age_name,year,val,upper,lower,PAF,scaler,smoke_cate,sdi_val,omega,actual_arc,arc,spline_pred,residuals,time_trend,mr_brt_val
0,1,Male,All ages,1990,4.039887,4.226359,3.858314,0.546926,2.207143,Mean Per Day,0.525767,0.0,1.348253,1.348253,4.222519,-0.182632,-0.077465,4.039887
1,2,Female,All ages,1990,4.404975,4.572757,4.226861,0.546926,2.207143,Mean Per Day,0.525767,0.0,1.133701,1.133701,4.222519,0.182456,-0.077465,4.404975
2,1,Male,All ages,1991,4.054218,4.240694,3.870502,0.546926,2.207143,Mean Per Day,0.530014,0.0,1.348253,1.348253,4.260208,-0.205990,-0.075257,4.054218
3,2,Female,All ages,1991,4.422290,4.590783,4.244331,0.546926,2.207143,Mean Per Day,0.530014,0.0,1.133701,1.133701,4.260208,0.162082,-0.075257,4.422290
4,1,Male,All ages,1992,4.067776,4.254614,3.881820,0.546926,2.207143,Mean Per Day,0.534287,0.0,1.348253,1.348253,4.256866,-0.189091,-0.073050,4.067776
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1087,2,Female,All ages,2049,4.958793,4.974764,4.602438,0.546926,2.207143,Mean Per Day,0.670289,2.5,1.133701,0.627877,4.618927,0.339866,-0.001658,4.290665
1088,1,Male,All ages,2050,4.263813,4.665762,4.275678,0.546926,2.207143,Mean Per Day,0.665228,2.5,1.348253,-0.691483,4.620178,-0.356365,-0.001730,4.291843
1089,1,Male,All ages,2050,4.263813,4.665762,4.275678,0.546926,2.207143,Mean Per Day,0.670289,2.5,1.348253,-0.691483,4.618927,-0.355114,-0.001730,4.290592
1090,2,Female,All ages,2050,4.965072,4.974764,4.602438,0.546926,2.207143,Mean Per Day,0.665228,2.5,1.133701,0.627877,4.620178,0.344893,-0.001730,4.291843


In [48]:
# 4. Apply PAF with omega
def apply_paf_without_limit(data, paf_column='PAF', omega_values=[0.0, 0.5, 1.0, 1.5, 2.0, 2.5], paf_scaling_factor=0.5):
    adjusted_data = []

    # Separate historical and forecast data
    historical_data = data[data['year'] <= 2021].copy()  # 실제 값 (1990-2021)
    forecast_data_raw = data[data['year'] >= 2022].copy()  # 예측 값 (2022-2050)

    # Use 'val' directly for historical data
    historical_data['final_prevalence'] = historical_data['val']  # 실제 값 유지

    # Apply PAF adjustments only to forecast data
    for omega in omega_values:
        data_omega = forecast_data_raw.loc[forecast_data_raw['omega'] == omega].copy()
        # PAF 값 제한 없이 그대로 사용
        data_omega['final_prevalence_raw'] = data_omega['mr_brt_val'] * (1 - data_omega[paf_column] * paf_scaling_factor)  # Adjusted forecast
        adjusted_data.append(data_omega)

    # Combine adjusted forecast data
    forecast_adjusted = pd.concat(adjusted_data, ignore_index=True)

    # Smooth transition: Adjust forecast to align with the last actual value (2021)
    last_actual_value = historical_data[historical_data['year'] == 2021]['final_prevalence'].mean()  # 2021년 최종 실제 값
    first_forecast_value = forecast_adjusted[forecast_adjusted['year'] == 2022]['final_prevalence_raw'].mean()  # 2022년 첫 예측 값

    scaling_factor = last_actual_value / first_forecast_value if first_forecast_value != 0 else 1.0

    # Scale forecast values for natural continuation
    forecast_adjusted['final_prevalence'] = forecast_adjusted['final_prevalence_raw'] * scaling_factor

    # Combine historical and forecast data
    final_data = pd.concat([historical_data, forecast_adjusted], ignore_index=True)
    return final_data


In [49]:
forecast_with_paf = apply_paf_without_limit(
    data=mr_brt_results_adjusted,
    paf_column='PAF',
    paf_scaling_factor=0.5
)

forecast_with_paf.to_csv('C:/Users/psy09/Desktop/Lab/5.AMD(GBD_database)/2.revision/prediction_model/data/forecast_with_paf.csv', index=False)
forecast_with_paf

,sex_id,sex_name,age_name,year,val,upper,lower,PAF,scaler,smoke_cate,sdi_val,omega,actual_arc,arc,spline_pred,residuals,time_trend,mr_brt_val,final_prevalence,final_prevalence_raw
0,1,Male,All ages,1990,4.039887,4.226359,3.858314,0.546926,2.207143,Mean Per Day,0.525767,0.0,1.348253,1.348253,4.222519,-0.182632,-0.077465,4.039887,4.039887,NaN
1,2,Female,All ages,1990,4.404975,4.572757,4.226861,0.546926,2.207143,Mean Per Day,0.525767,0.0,1.133701,1.133701,4.222519,0.182456,-0.077465,4.404975,4.404975,NaN
2,1,Male,All ages,1991,4.054218,4.240694,3.870502,0.546926,2.207143,Mean Per Day,0.530014,0.0,1.348253,1.348253,4.260208,-0.205990,-0.075257,4.054218,4.054218,NaN
3,2,Female,All ages,1991,4.422290,4.590783,4.244331,0.546926,2.207143,Mean Per Day,0.530014,0.0,1.133701,1.133701,4.260208,0.162082,-0.075257,4.422290,4.422290,NaN
4,1,Male,All ages,1992,4.067776,4.254614,3.881820,0.546926,2.207143,Mean Per Day,0.534287,0.0,1.348253,1.348253,4.256866,-0.189091,-0.073050,4.067776,4.067776,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1087,2,Female,All ages,2049,4.958793,4.974764,4.602438,0.546926,2.207143,Mean Per Day,0.670289,2.5,1.133701,0.627877,4.618927,0.339866,-0.001658,4.290665,4.475670,3.117328
1088,1,Male,All ages,2050,4.263813,4.665762,4.275678,0.546926,2.207143,Mean Per Day,0.665228,2.5,1.348253,-0.691483,4.620178,-0.356365,-0.001730,4.291843,4.476899,3.118184
1089,1,Male,All ages,2050,4.263813,4.665762,4.275678,0.546926,2.207143,Mean Per Day,0.670289,2.5,1.348253,-0.691483,4.618927,-0.355114,-0.001730,4.290592,4.475593,3.117275
1090,2,Female,All ages,2050,4.965072,4.974764,4.602438,0.546926,2.207143,Mean Per Day,0.665228,2.5,1.133701,0.627877,4.620178,0.344893,-0.001730,4.291843,4.476899,3.118184


In [50]:
# 5. RMSE 계산
epsilon = 1e-6  # 작은 상수
rmse_values = {}

# 각 omega에 대해 RMSE 계산 (2010~2019 데이터 사용)
for omega in forecast_with_paf['omega'].unique():
    validation_data = forecast_with_paf[
        (forecast_with_paf['year'] >= 2010) & 
        (forecast_with_paf['year'] <= 2019) & 
        (forecast_with_paf['omega'] == omega)
    ].dropna(subset=['val', 'mr_brt_val'])  # NaN 제거

    if not validation_data.empty:
        rmse = np.sqrt(np.mean((validation_data['val'] - validation_data['mr_brt_val'])**2))
        rmse_values[omega] = rmse
    else:
        rmse_values[omega] = None  # 유효하지 않은 경우 None 설정

# 유효한 RMSE 값만 사용
valid_rmse_values = {omega: rmse for omega, rmse in rmse_values.items() if rmse is not None}

In [51]:
# 6. Ensemble Calculation
if valid_rmse_values:
    best_omega = min(valid_rmse_values, key=valid_rmse_values.get)
    print(f"Selected omega with minimum RMSE: {best_omega}")
else:
    raise ValueError("No valid RMSE values available for omega selection.")

final_data = []

for omega in forecast_with_paf['omega'].unique():
    # omega별로 데이터 필터링
    filtered_data = forecast_with_paf[forecast_with_paf['omega'] == omega]

    for (sex, age, year, smoke_cate), group in filtered_data.groupby(
        ['sex_name', 'age_name', 'year', 'smoke_cate']
    ):
        if group.empty:
            print(f"Warning: Empty group for sex={sex}, age={age}, year={year}, smoke_cate={smoke_cate}, omega={omega}.")
            arc_model = 0
            mrbrt_arc_ensemble = 0
            finalprev_arc_ensemble = 0
        else:
            # ARC 모델 (val 사용)
            arc_model = group['val'].mean()  # ARC 모델 값 (val 칼럼)

            # MR-BRT와 ARC의 앙상블
            mrbrt_arc_ensemble = 0.5 * group['mr_brt_val'].mean() + 0.5 * arc_model

            # Final Prevalence와 ARC의 앙상블
            finalprev_arc_ensemble = 0.5 * group['final_prevalence'].mean() + 0.5 * arc_model

        # 결과 저장
        final_data.append({
            'sex_name': sex,
            'age_name': age,
            'year': year,
            'smoke_cate': smoke_cate,
            'omega': omega,  # omega 값 포함
            'arc_model': arc_model,  # ARC 모델 (val 값 기반)
            'mrbrt_arc_ensemble': mrbrt_arc_ensemble,  # MR-BRT와 ARC의 앙상블
            'finalprev_arc_ensemble': finalprev_arc_ensemble  # Final Prevalence와 ARC의 앙상블
        })

# 최종 데이터프레임 생성
final_ensemble_df = pd.DataFrame(final_data)

Selected omega with minimum RMSE: 0.0


In [52]:
final_ensemble_df = pd.DataFrame(final_data)
final_ensemble_df.to_csv('C:/Users/psy09/Desktop/Lab/5.AMD(GBD_database)/2.revision/prediction_model/data/final_ensemble_df.csv', index=False)
final_ensemble_df

,sex_name,age_name,year,smoke_cate,omega,arc_model,mrbrt_arc_ensemble,finalprev_arc_ensemble
0,Female,All ages,1990,Mean Per Day,0.0,4.404975,4.404975,4.404975
1,Female,All ages,1991,Mean Per Day,0.0,4.422290,4.422290,4.422290
2,Female,All ages,1992,Mean Per Day,0.0,4.437326,4.437326,4.437326
3,Female,All ages,1993,Mean Per Day,0.0,4.450161,4.450161,4.450161
4,Female,All ages,1994,Mean Per Day,0.0,4.460874,4.460874,4.460874
...,...,...,...,...,...,...,...,...
727,Male,All ages,2046,Mean Per Day,2.5,4.291473,4.291491,4.384012
728,Male,All ages,2047,Mean Per Day,2.5,4.284558,4.287997,4.380516
729,Male,All ages,2048,Mean Per Day,2.5,4.277643,4.284503,4.377021
730,Male,All ages,2049,Mean Per Day,2.5,4.270728,4.281009,4.373525


In [53]:
# 95% UI 
def calculate_ui(group):
    if len(group) > 1:  # If group size > 1, calculate std
        std_arc_mrbrt = group['mrbrt_arc_ensemble'].std()
        std_arc_final = group['finalprev_arc_ensemble'].std()
    else:
        # If group size is 1, std is 0
        std_arc_mrbrt = 0
        std_arc_final = 0

    group['mrbrt_arc_ensemble_lower'] = group['mrbrt_arc_ensemble'] - 1.96 * std_arc_mrbrt
    group['mrbrt_arc_ensemble_upper'] = group['mrbrt_arc_ensemble'] + 1.96 * std_arc_mrbrt
    group['finalprev_arc_ensemble_lower'] = group['finalprev_arc_ensemble'] - 1.96 * std_arc_final
    group['finalprev_arc_ensemble_upper'] = group['finalprev_arc_ensemble'] + 1.96 * std_arc_final
    return group

In [61]:
final_forecast = pd.DataFrame(final_ensemble_df)
forecast_final_ui = final_forecast.groupby(['sex_name', 'age_name', 'smoke_cate'], group_keys=False).apply(calculate_ui)

column_order = [
    'sex_name', 'age_name', 'year', 'smoke_cate', 'omega',
    'mrbrt_arc_ensemble', 'mrbrt_arc_ensemble_lower', 'mrbrt_arc_ensemble_upper',
    'finalprev_arc_ensemble', 'finalprev_arc_ensemble_lower', 'finalprev_arc_ensemble_upper'
]
forecast_final_ui = forecast_final_ui[column_order]

C:\Users\psy09\AppData\Local\Temp\ipykernel_22620\706999760.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  forecast_final_ui = final_forecast.groupby(['sex_name', 'age_name', 'smoke_cate'], group_keys=False).apply(calculate_ui)


In [62]:
# additional..ui
condition = (forecast_final_ui['year'] >= 1990) & (forecast_final_ui['year'] <= 2021)
forecast_final_ui.loc[condition, 'finalprev_arc_ensemble'] = forecast_final_ui.loc[condition, 'mrbrt_arc_ensemble']
forecast_final_ui.loc[condition, 'finalprev_arc_ensemble_lower'] = forecast_final_ui.loc[condition, 'mrbrt_arc_ensemble_lower']
forecast_final_ui.loc[condition, 'finalprev_arc_ensemble_upper'] = forecast_final_ui.loc[condition, 'mrbrt_arc_ensemble_upper']

forecast_final_ui.to_csv('C:/Users/psy09/Desktop/Lab/5.AMD(GBD_database)/2.revision/prediction_model/data/forecast_final_ui.csv', index=False)
forecast_final_ui

,sex_name,age_name,year,smoke_cate,omega,mrbrt_arc_ensemble,mrbrt_arc_ensemble_lower,mrbrt_arc_ensemble_upper,finalprev_arc_ensemble,finalprev_arc_ensemble_lower,finalprev_arc_ensemble_upper
0,Female,All ages,1990,Mean Per Day,0.0,4.404975,4.145391,4.664559,4.404975,4.145391,4.664559
1,Female,All ages,1991,Mean Per Day,0.0,4.422290,4.162706,4.681874,4.422290,4.162706,4.681874
2,Female,All ages,1992,Mean Per Day,0.0,4.437326,4.177742,4.696909,4.437326,4.177742,4.696909
3,Female,All ages,1993,Mean Per Day,0.0,4.450161,4.190577,4.709745,4.450161,4.190577,4.709745
4,Female,All ages,1994,Mean Per Day,0.0,4.460874,4.201290,4.720458,4.460874,4.201290,4.720458
...,...,...,...,...,...,...,...,...,...,...,...
727,Male,All ages,2046,Mean Per Day,2.5,4.291491,3.916112,4.666870,4.384012,3.933456,4.834567
728,Male,All ages,2047,Mean Per Day,2.5,4.287997,3.912618,4.663376,4.380516,3.929961,4.831071
729,Male,All ages,2048,Mean Per Day,2.5,4.284503,3.909124,4.659882,4.377021,3.926465,4.827576
730,Male,All ages,2049,Mean Per Day,2.5,4.281009,3.905631,4.656388,4.373525,3.922970,4.824081
